In [1]:
config = {
    "vocab_size": 32000,
    "seq_len": 128,
    "n_layers": 4,
    "n_heads": 4,
    "d_model": 256,
    "dropout": 0.1,
}

In [2]:
import torch
import torch.nn as nn
import math

class Head(nn.Module):
    def __init__(self, d_model, head_size):
        super().__init__()

        self.key = nn.Linear(d_model, head_size, bias=False)
        self.query = nn.Linear(d_model, head_size, bias=False)
        self.value = nn.Linear(d_model, head_size, bias=False)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)

        scores = q @ k.transpose(-2, -1)
        scores = scores / math.sqrt(k.size(-1))

        weights = torch.softmax(scores, dim=-1)

        v = self.value(x)

        out = weights @ v

        return out

In [3]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        head_size = d_model // n_heads

        self.heads = nn.ModuleList([
            Head(d_model, head_size)
            for _ in range(n_heads)
        ])

        self.proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        out = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )

        return self.proj(out)

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model)
        )

    def forward(self, x):
        return self.net(x)

In [5]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = FeedForward(d_model)

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [6]:
class GPT(nn.Module):
    def __init__(
        self,
        vocab_size,
        seq_len,
        d_model,
        n_heads,
        n_layers
    ):
        super().__init__()

        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        self.position_embedding = nn.Embedding(
            seq_len,
            d_model
        )

        self.blocks = nn.Sequential(
            *[
                Block(d_model, n_heads)
                for _ in range(n_layers)
            ]
        )

        self.ln_f = nn.LayerNorm(d_model)

        self.head = nn.Linear(
            d_model,
            vocab_size
        )

    def forward(self, idx):
        B, T = idx.shape

        tok_emb = self.token_embedding(idx)

        pos = torch.arange(T, device=idx.device)

        pos_emb = self.position_embedding(pos)

        x = tok_emb + pos_emb

        x = self.blocks(x)

        x = self.ln_f(x)

        logits = self.head(x)

        return logits

In [7]:
model = GPT(
    vocab_size=32000,
    seq_len=128,
    d_model=256,
    n_heads=4,
    n_layers=4
)

params = sum(
    p.numel()
    for p in model.parameters()
)

print(params)

19605248


In [8]:
device = "cuda"

model = model.to(device)

AssertionError: Torch not compiled with CUDA enabled